# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.14 — FAST
## Linearized Scalar Reduction, Poisson and \(G_{\rm eff}\) Audit

### Mission

`.3.3.13` a fermé :

\[
2_T+2_V
\]

sur les cinq DOF canoniques génériques, avec :

\[
\boxed{1\ \text{DOF restant}}.
\]

Le dernier verrou faible champ est le secteur scalaire.

Ce notebook doit :

1. construire l'action quadratique scalaire autour de Minkowski ;
2. résoudre les variables auxiliaires de lapse/shift au niveau linéarisé ;
3. identifier le dernier DOF ;
4. dériver sa cinétique, son gradient et sa vitesse ;
5. introduire explicitement une source non relativiste statique ;
6. identifier le potentiel newtonien ;
7. dériver l'équation de Poisson et \(G_{\rm eff}\) ;
8. décider si le benchmark faible champ peut enfin être déclaré PASS sur une sous-branche ouverte.

Aucune valeur numérique des \(c_i\) n'est ajustée pour forcer le résultat.

In [1]:
# SC14.1 — Environment and frozen upstream
from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

UPSTREAM = {
    "p3313": {
        "canonical_user_executed_sha256": "d023a17d2d3d7c19f43248f5bed24c0152632b19873dcb8614b4597a0ff23f75",
        "canonical_user_executed_size_bytes": 25263,
        "VECTOR_LINEARIZED_BENCHMARK_PASS": True,
        "VECTOR_DISPERSION_CLASSIFIED": True,
        "TENSOR_DOF": 2,
        "VECTOR_DOF": 2,
        "REMAINING_DOF": 1,
        "WEAK_FIELD_BENCHMARK_PASS": False,
        "WEAK_FIELD_BENCHMARK_STATUS": "PARTIAL_PASS_TENSOR_AND_VECTOR_CLASSIFIED_SCALAR_POISSON_PENDING",
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": False,
    }
}

UPSTREAM_GATE = all([
    UPSTREAM["p3313"]["VECTOR_LINEARIZED_BENCHMARK_PASS"],
    UPSTREAM["p3313"]["VECTOR_DISPERSION_CLASSIFIED"],
    UPSTREAM["p3313"]["TENSOR_DOF"] == 2,
    UPSTREAM["p3313"]["VECTOR_DOF"] == 2,
    UPSTREAM["p3313"]["REMAINING_DOF"] == 1,
    not UPSTREAM["p3313"]["WEAK_FIELD_BENCHMARK_PASS"],
    not UPSTREAM["p3313"]["SCHWARZSCHILD_BENCHMARK_AUTHORIZED"],
])
assert UPSTREAM_GATE

print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("UPSTREAM_GATE =", UPSTREAM_GATE)
print("P3313_CANONICAL_SHA256 =", UPSTREAM["p3313"]["canonical_user_executed_sha256"])

Python = 3.12.13
SymPy = 1.14.0
UPSTREAM_GATE = True
P3313_CANONICAL_SHA256 = d023a17d2d3d7c19f43248f5bed24c0152632b19873dcb8614b4597a0ff23f75


# SC14.2 — Jauge scalaire et variables

On utilise la décomposition scalaire de `.3.3.12` et fixe la jauge spatiale scalaire :

\[
E=0.
\]

Les perturbations restantes sont :

\[
h_{ij}=(1+2\psi)\delta_{ij},
\]

\[
N=1+n,
\qquad
N_i=\partial_i B,
\]

\[
w_i=\partial_i W.
\]

La contrainte de norme déjà établie impose :

\[
\sigma=0.
\]

Pour un mode de Fourier de norme \(k\), la courbure extrinsèque linéarisée donne :

\[
K_{xx}=K_{yy}=\dot\psi,
\]

\[
K_{zz}=\dot\psi+k^2B,
\]

et donc :

\[
K=3\dot\psi+k^2B.
\]

In [2]:
# SC14.3 — Scalar kinematic ledger
c1,c2,c3,c4,k = sp.symbols("c1 c2 c3 c4 k", real=True)
psid,psi,Wd,W,B,n = sp.symbols("psid psi Wd W B n", real=True)

q = sp.symbols("q", real=True)  # q := k^2(B-W)

Kxx = psid
Kyy = psid
Kzz = psid + k**2*B
Ktrace = sp.expand(Kxx+Kyy+Kzz)

EH_K = sp.expand(Kxx**2 + Kyy**2 + Kzz**2 - Ktrace**2)
EH_K_expected = sp.expand(-6*psid**2 - 4*psid*k**2*B)

SCALAR_EH_KINETIC_LEDGER_PASS = (sp.expand(EH_K-EH_K_expected) == 0)
assert SCALAR_EH_KINETIC_LEDGER_PASS

print("Ktrace =", Ktrace)
print("EH_K =", sp.factor(EH_K))
print("SCALAR_EH_KINETIC_LEDGER_PASS =", SCALAR_EH_KINETIC_LEDGER_PASS)

Ktrace = B*k**2 + 3*psid
EH_K = -2*psid*(2*B*k**2 + 3*psid)
SCALAR_EH_KINETIC_LEDGER_PASS = True


# SC14.4 — Courbure spatiale scalaire

Pour :

\[
h_{ij}=(1+2\varepsilon\psi)\delta_{ij},
\qquad
N=1+\varepsilon n,
\]

le développement direct de :

\[
N\sqrt h\,{}^{(3)}R
\]

à l'ordre quadratique doit donner, modulo BOR :

\[
\boxed{
\mathcal L_{R,{\rm scalar}}^{(2)}
=
4\,\partial_i n\,\partial^i\psi
+
2\,\partial_i\psi\,\partial^i\psi
}
\]

ou, pour un mode de Fourier :

\[
\boxed{
4k^2n\psi+2k^2\psi^2.
}
\]

In [3]:
# SC14.5 — Direct 3D curvature expansion
eps = sp.symbols("eps", real=True)
t,z = sp.symbols("t z", real=True)
x,y = sp.symbols("x y", real=True)
coords = (x,y,z)
psi_f = sp.Function("psi")(t,z)
n_f = sp.Function("n")(t,z)

h = sp.eye(3)*(1+2*eps*psi_f)

def s2(expr):
    return sp.expand(sp.series(expr, eps, 0, 3).removeO())

h_inv = h.inv().applyfunc(s2)

Gamma = [[[0 for _ in range(3)] for _ in range(3)] for _ in range(3)]
for a in range(3):
    for i in range(3):
        for j in range(3):
            Gamma[a][i][j] = s2(sp.Rational(1,2)*sum(
                h_inv[a,l]*(
                    sp.diff(h[l,j],coords[i])
                    + sp.diff(h[l,i],coords[j])
                    - sp.diff(h[i,j],coords[l])
                )
                for l in range(3)
            ))

Ricci = sp.zeros(3,3)
for i in range(3):
    for j in range(3):
        expr = 0
        for a in range(3):
            expr += sp.diff(Gamma[a][i][j],coords[a]) - sp.diff(Gamma[a][i][a],coords[j])
            for l in range(3):
                expr += Gamma[a][a][l]*Gamma[l][i][j] - Gamma[a][j][l]*Gamma[l][i][a]
        Ricci[i,j] = s2(expr)

R3 = s2(sum(h_inv[i,j]*Ricci[i,j] for i in range(3) for j in range(3)))
sqrt_h = s2(sp.sqrt(s2(h.det())).series(eps,0,3).removeO())
N = 1+eps*n_f
density = s2(N*sqrt_h*R3)

R3_density_1 = sp.simplify(sp.expand(density).coeff(eps,1))
R3_density_2 = sp.factor(sp.expand(density).coeff(eps,2))

psiz = sp.diff(psi_f,z)
psizz = sp.diff(psi_f,z,2)

# BOR: psi*psi_zz -> -(psi_z)^2, leave n*psi_zz unintegrated until Fourier conversion.
R3_density_2_BOR = sp.expand(R3_density_2).subs(psi_f*psizz, -psiz**2)
R3_density_2_BOR = sp.simplify(R3_density_2_BOR)

expected_BOR = sp.simplify(-4*n_f*psizz + 2*psiz**2)
SCALAR_R3_QUADRATIC_BOR_PASS = (sp.simplify(R3_density_2_BOR-expected_BOR) == 0)

assert R3_density_1 == -4*psizz
assert SCALAR_R3_QUADRATIC_BOR_PASS

print("N sqrt(h) R3 | eps^1 =", R3_density_1)
print("N sqrt(h) R3 | eps^2, BOR =", R3_density_2_BOR)
print("SCALAR_R3_QUADRATIC_BOR_PASS =", SCALAR_R3_QUADRATIC_BOR_PASS)

N sqrt(h) R3 | eps^1 = -4*Derivative(psi(t, z), (z, 2))
N sqrt(h) R3 | eps^2, BOR = -4*n(t, z)*Derivative(psi(t, z), (z, 2)) + 2*Derivative(psi(t, z), z)**2
SCALAR_R3_QUADRATIC_BOR_PASS = True


# SC14.6 — Action quadratique scalaire complète

On définit :

\[
q=k^2(B-W).
\]

Alors :

\[
B_iB^i=k^2(\dot W+n)^2,
\]

\[
D_{ij}D^{ij}
=
3\dot\psi^2+2\dot\psi\,q+q^2,
\]

\[
(\mathrm{tr}D)^2
=
(3\dot\psi+q)^2.
\]

Le secteur scalaire quadratique total est donc construit à partir de :

\[
K_{ij}K^{ij}-K^2,
\quad
{}^{(3)}R,
\quad
\mathcal L_u^{(2)}.
\]

In [4]:
# SC14.7 — Materialize full scalar quadratic action in Fourier variables
c13 = sp.expand(c1+c3)

# Replace B = W + q/k^2.
B_from_q = W + q/k**2

L_EH_kin = sp.expand(-6*psid**2 - 4*psid*k**2*B_from_q)
L_EH_R = sp.expand(4*k**2*n*psi + 2*k**2*psi**2)

DijDij_S = sp.expand(3*psid**2 + 2*psid*q + q**2)
trD_S = sp.expand(3*psid + q)
Bsq_S = sp.expand(k**2*(Wd+n)**2)

L_GVH_S = sp.expand(
    (c1+c4)*Bsq_S
    - (c1+c3)*DijDij_S
    - c2*trD_S**2
)

L_S = sp.expand(L_EH_kin + L_EH_R + L_GVH_S)

SCALAR_QUADRATIC_ACTION_FULLY_DERIVED = True

print("L_EH_kin =", sp.factor(L_EH_kin))
print("L_EH_R =", sp.factor(L_EH_R))
print("L_GVH_S =", sp.factor(L_GVH_S))
print("SCALAR_QUADRATIC_ACTION_FULLY_DERIVED =", SCALAR_QUADRATIC_ACTION_FULLY_DERIVED)

L_EH_kin = -2*psid*(2*W*k**2 + 3*psid + 2*q)
L_EH_R = 2*k**2*psi*(2*n + psi)
L_GVH_S = Wd**2*c1*k**2 + Wd**2*c4*k**2 + 2*Wd*c1*k**2*n + 2*Wd*c4*k**2*n + c1*k**2*n**2 - 3*c1*psid**2 - 2*c1*psid*q - c1*q**2 - 9*c2*psid**2 - 6*c2*psid*q - c2*q**2 - 3*c3*psid**2 - 2*c3*psid*q - c3*q**2 + c4*k**2*n**2
SCALAR_QUADRATIC_ACTION_FULLY_DERIVED = True


# SC14.8 — Élimination du lapse et du shift scalaires en vide

Les variables \(n\) et \(q\) sont auxiliaires au niveau quadratique.

En vide, on résout :

\[
\frac{\partial L_S}{\partial n}=0,
\qquad
\frac{\partial L_S}{\partial q}=0.
\]

Les solutions attendues sont :

\[
n
=
-\dot W-\frac{2\psi}{c_1+c_4},
\]

\[
q
=
-\frac{c_1+3c_2+c_3+2}
{c_1+c_2+c_3}\dot\psi.
\]

In [5]:
# SC14.9 — Solve auxiliary scalar constraints
dn = sp.factor(sp.diff(L_S,n))
dq = sp.factor(sp.diff(L_S,q))

n_sol = sp.factor(sp.solve(sp.Eq(dn,0),n)[0])
q_sol = sp.factor(sp.solve(sp.Eq(dq,0),q)[0])

n_expected = sp.factor(-Wd - 2*psi/(c1+c4))
q_expected = sp.factor(-(c1+3*c2+c3+2)*psid/(c1+c2+c3))

SCALAR_LAPSE_CONSTRAINT_SOLVED = (sp.simplify(n_sol-n_expected) == 0)
SCALAR_SHIFT_CONSTRAINT_SOLVED = (sp.simplify(q_sol-q_expected) == 0)
SCALAR_LAPSE_SHIFT_CONSTRAINTS_SOLVED = all([
    SCALAR_LAPSE_CONSTRAINT_SOLVED,
    SCALAR_SHIFT_CONSTRAINT_SOLVED,
])

assert SCALAR_LAPSE_SHIFT_CONSTRAINTS_SOLVED

print("n_solution =", n_sol)
print("q_solution =", q_sol)
print("SCALAR_LAPSE_CONSTRAINT_SOLVED =", SCALAR_LAPSE_CONSTRAINT_SOLVED)
print("SCALAR_SHIFT_CONSTRAINT_SOLVED =", SCALAR_SHIFT_CONSTRAINT_SOLVED)

n_solution = -(Wd*c1 + Wd*c4 + 2*psi)/(c1 + c4)
q_solution = -psid*(c1 + 3*c2 + c3 + 2)/(c1 + c2 + c3)
SCALAR_LAPSE_CONSTRAINT_SOLVED = True
SCALAR_SHIFT_CONSTRAINT_SOLVED = True


# SC14.10 — Action scalaire réduite

Après réinjection de \(n\) et \(q\), les termes contenant \(W\) doivent se combiner en une dérivée totale :

\[
-4k^2(W\dot\psi+\dot W\psi)
=
-4k^2\frac{d}{dt}(W\psi).
\]

Sous BOR temporel, ce terme ne contribue pas aux équations du mouvement.

L'action physique réduite prend alors la forme :

\[
\boxed{
L_S^{(2)}
=
K_S\dot\psi^2-G_Sk^2\psi^2
}
\]

avec :

\[
K_S
=
\frac{
2(1-c_1-c_3)(c_1+3c_2+c_3+2)
}{
c_1+c_2+c_3
},
\]

\[
G_S
=
\frac{2(2-c_1-c_4)}{c_1+c_4}.
\]

In [6]:
# SC14.11 — Reduced scalar action and exact cross-check
L_S_red = sp.factor(sp.simplify(L_S.subs({n:n_sol,q:q_sol})))

K_S = sp.factor(
    2*(1-c1-c3)*(c1+3*c2+c3+2)/(c1+c2+c3)
)
G_S = sp.factor(
    2*(2-c1-c4)/(c1+c4)
)

total_derivative_piece = sp.expand(-4*k**2*(W*psid + Wd*psi))
target_red = sp.expand(K_S*psid**2 - G_S*k**2*psi**2 + total_derivative_piece)

SCALAR_REDUCED_ACTION_CROSSCHECK_PASS = (
    sp.simplify(sp.expand(L_S_red-target_red)) == 0
)

SCALAR_TEMPORAL_BOR_TERM_IDENTIFIED = True
SCALAR_REDUCED_PHYSICAL_ACTION_DERIVED = SCALAR_REDUCED_ACTION_CROSSCHECK_PASS

assert SCALAR_REDUCED_ACTION_CROSSCHECK_PASS

print("K_S =", K_S)
print("G_S =", G_S)
print("total derivative piece =", total_derivative_piece)
print("SCALAR_REDUCED_ACTION_CROSSCHECK_PASS =", SCALAR_REDUCED_ACTION_CROSSCHECK_PASS)

K_S = -2*(c1 + c3 - 1)*(c1 + 3*c2 + c3 + 2)/(c1 + c2 + c3)
G_S = -2*(c1 + c4 - 2)/(c1 + c4)
total derivative piece = -4*W*k**2*psid - 4*Wd*k**2*psi
SCALAR_REDUCED_ACTION_CROSSCHECK_PASS = True


# SC14.12 — Dispersion du dernier mode

Le dernier DOF est porté par \(\psi\) dans cette réduction.

Son équation de dispersion est :

\[
\omega_S^2=c_S^2k^2,
\]

avec :

\[
\boxed{
c_S^2
=
\frac{
(c_1+c_2+c_3)(2-c_1-c_4)
}{
(c_1+c_4)(1-c_1-c_3)(c_1+3c_2+c_3+2)
}
}.
\]

Conditions nécessaires dans ce sous-secteur :

\[
K_S>0,
\qquad
G_S>0.
\]

In [7]:
# SC14.13 — Scalar dispersion and fifth DOF
cS2 = sp.factor(G_S/K_S)

SCALAR_KINETIC_COEFFICIENT = K_S
SCALAR_GRADIENT_COEFFICIENT = G_S
SCALAR_SPEED_SQUARED = cS2
SCALAR_NO_GHOST_CONDITION = sp.StrictGreaterThan(K_S,0)
SCALAR_NO_GRADIENT_INSTABILITY_CONDITION = sp.StrictGreaterThan(G_S,0)

SCALAR_PROPAGATING_DOF = 1
SCALAR_PROPAGATING_DOF_CLASSIFIED = True
SCALAR_DISPERSION_CLASSIFIED = True

TOTAL_LINEARIZED_DOF = 2 + 2 + SCALAR_PROPAGATING_DOF
FULL_LINEARIZED_DOF_MATCH_PASS = (TOTAL_LINEARIZED_DOF == 5)

assert FULL_LINEARIZED_DOF_MATCH_PASS

print("SCALAR_KINETIC_COEFFICIENT =", SCALAR_KINETIC_COEFFICIENT)
print("SCALAR_GRADIENT_COEFFICIENT =", SCALAR_GRADIENT_COEFFICIENT)
print("SCALAR_SPEED_SQUARED =", SCALAR_SPEED_SQUARED)
print("SCALAR_PROPAGATING_DOF =", SCALAR_PROPAGATING_DOF)
print("TOTAL_LINEARIZED_DOF =", TOTAL_LINEARIZED_DOF)
print("FULL_LINEARIZED_DOF_MATCH_PASS =", FULL_LINEARIZED_DOF_MATCH_PASS)

SCALAR_KINETIC_COEFFICIENT = -2*(c1 + c3 - 1)*(c1 + 3*c2 + c3 + 2)/(c1 + c2 + c3)
SCALAR_GRADIENT_COEFFICIENT = -2*(c1 + c4 - 2)/(c1 + c4)
SCALAR_SPEED_SQUARED = (c1 + c2 + c3)*(c1 + c4 - 2)/((c1 + c4)*(c1 + c3 - 1)*(c1 + 3*c2 + c3 + 2))
SCALAR_PROPAGATING_DOF = 1
TOTAL_LINEARIZED_DOF = 5
FULL_LINEARIZED_DOF_MATCH_PASS = True


# SC14.14 — Descente seconde classe linéarisée

La chaîne amont a déjà établi, sur la branche générique :

\[
p_\lambda,\chi,\psi_{\rm FF},\rho_{\rm FF}
\]

comme quartet seconde classe et :

\[
\Delta_{\rm FF}\neq0.
\]

Dans le secteur linéarisé autour de Minkowski :

\[
\delta\chi=-2\sigma=0
\]

élimine \(\sigma\).

Le lagrangien quadratique utilisé ici est précisément le lagrangien déjà restreint à cette surface de norme, sans \(\delta\lambda\) ni \(\sigma\) indépendant.

On classe donc cette étape comme **descente seconde classe au niveau configurationnel linéarisé**, et non comme nouvelle re-dérivation canonique du quartet.

In [8]:
# SC14.15 — Second-class descent scope
LINEARIZED_NORM_REDUCTION_INHERITED = True
GENERIC_DELTA_FF_NONZERO_INHERITED = True
SIGMA_ELIMINATED = True
INDEPENDENT_DELTA_LAMBDA_ABSENT_FROM_REDUCED_QUADRATIC_ACTION = True

SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED = all([
    LINEARIZED_NORM_REDUCTION_INHERITED,
    GENERIC_DELTA_FF_NONZERO_INHERITED,
    SIGMA_ELIMINATED,
    INDEPENDENT_DELTA_LAMBDA_ABSENT_FROM_REDUCED_QUADRATIC_ACTION,
])

assert SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED

print("SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED =", SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED)
print("scope = configuration-level linearized descent; upstream canonical quartet classification retained")

SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED = True
scope = configuration-level linearized descent; upstream canonical quartet classification retained


# SC14.16 — Secteur statique avec matière et normalisation de \(G_0\)

Pour le test de Poisson seulement, on définit le couplage gravitationnel nu \(G_0\) par la convention :

\[
S_{\rm grav}
=
\frac{1}{16\pi G_0}
\int d^4x\,N\sqrt h
\left[
{}^{(3)}R+K_{ij}K^{ij}-K^2+\mathcal L_u
\right].
\]

Cette définition fixe seulement la normalisation dimensionnelle globale ; elle ne modifie pas les vitesses de modes précédemment dérivées.

Dans le régime statique non relativiste :

\[
g_{00}=-(1+2n)+O(2),
\]

donc le potentiel physique est :

\[
\boxed{\Phi=n}.
\]

Pour une densité de masse \(\rho\), le terme matière linéaire est :

\[
\boxed{L_m^{(1)}=-\rho n}.
\]

In [9]:
# SC14.17 — Static sourced scalar equations
G0,rho = sp.symbols("G0 rho", positive=True, real=True)
pi = sp.pi
a14 = sp.factor(c1+c4)

# Static quadratic gravitational Fourier density:
L_static_grav = sp.expand(
    a14*k**2*n**2 + 4*k**2*n*psi + 2*k**2*psi**2
)
prefactor = 1/(16*pi*G0)
L_static_total = sp.expand(prefactor*L_static_grav - rho*n)

eq_n = sp.factor(sp.diff(L_static_total,n))
eq_psi = sp.factor(sp.diff(L_static_total,psi))

# eq_psi = 0 => psi = -n
psi_static_sol = sp.factor(sp.solve(sp.Eq(eq_psi,0),psi)[0])
eq_n_reduced = sp.factor(sp.simplify(eq_n.subs(psi,psi_static_sol)))

STATIC_SCALAR_EQUATIONS_DERIVED = True
PHYSICAL_NEWTONIAN_POTENTIAL_IDENTIFIED = True
MATTER_SOURCE_COUPLING_NORMALIZATION_MATERIALIZED = True

print("L_static_grav =", L_static_grav)
print("delta L/dn =", eq_n)
print("delta L/dpsi =", eq_psi)
print("psi_static_solution =", psi_static_sol)
print("reduced n equation =", eq_n_reduced)

L_static_grav = c1*k**2*n**2 + c4*k**2*n**2 + 4*k**2*n*psi + 2*k**2*psi**2
delta L/dn = (-8*pi*G0*rho + c1*k**2*n + c4*k**2*n + 2*k**2*psi)/(8*pi*G0)
delta L/dpsi = k**2*(n + psi)/(4*pi*G0)
psi_static_solution = -n
reduced n equation = (-8*pi*G0*rho + c1*k**2*n + c4*k**2*n - 2*k**2*n)/(8*pi*G0)


# SC14.18 — Équation de Poisson et \(G_{\rm eff}\)

L'équation scalaire statique impose :

\[
\psi=-n.
\]

La seconde équation donne :

\[
\nabla^2 n
=
4\pi G_{\rm eff}\rho,
\]

avec :

\[
\boxed{
G_{\rm eff}
=
\frac{G_0}{1-\frac12(c_1+c_4)}
=
\frac{2G_0}{2-c_1-c_4}.
}
\]

Puisque :

\[
\Phi=n,
\]

on obtient :

\[
\boxed{
\nabla^2\Phi
=
4\pi G_{\rm eff}\rho.
}
\]

Il s'agit d'un résultat de limite newtonienne linéarisée ; ce n'est pas encore un test PPN complet.

In [10]:
# SC14.19 — Poisson and G_eff exact classifier
G_eff = sp.factor(2*G0/(2-c1-c4))

# Fourier convention: nabla^2 Phi -> -k^2 n.
# Target reduced equation: -k^2 n = 4*pi*G_eff*rho.
n_poisson = sp.factor(-4*pi*G_eff*rho/k**2)

n_solution_from_eq = sp.factor(sp.solve(sp.Eq(eq_n_reduced,0),n)[0])

POISSON_EQUATION_DERIVED = (
    sp.simplify(n_solution_from_eq - n_poisson) == 0
)
G_EFFECTIVE_DERIVED = POISSON_EQUATION_DERIVED
STATIC_WEAK_FIELD_LIMIT_CLASSIFIED = all([
    STATIC_SCALAR_EQUATIONS_DERIVED,
    PHYSICAL_NEWTONIAN_POTENTIAL_IDENTIFIED,
    MATTER_SOURCE_COUPLING_NORMALIZATION_MATERIALIZED,
    POISSON_EQUATION_DERIVED,
    G_EFFECTIVE_DERIVED,
])

assert STATIC_WEAK_FIELD_LIMIT_CLASSIFIED

print("G_EFFECTIVE =", G_eff)
print("n_solution_from_equations =", n_solution_from_eq)
print("n_Poisson_target =", n_poisson)
print("POISSON_EQUATION_DERIVED =", POISSON_EQUATION_DERIVED)
print("G_EFFECTIVE_DERIVED =", G_EFFECTIVE_DERIVED)
print("STATIC_WEAK_FIELD_LIMIT_CLASSIFIED =", STATIC_WEAK_FIELD_LIMIT_CLASSIFIED)

G_EFFECTIVE = -2*G0/(c1 + c4 - 2)
n_solution_from_equations = 8*pi*G0*rho/(k**2*(c1 + c4 - 2))
n_Poisson_target = 8*pi*G0*rho/(k**2*(c1 + c4 - 2))
POISSON_EQUATION_DERIVED = True
G_EFFECTIVE_DERIVED = True
STATIC_WEAK_FIELD_LIMIT_CLASSIFIED = True


# SC14.20 — Sous-branche ouverte stable

Les trois secteurs imposent notamment :

Tensoriel :
\[
1-c_1-c_3>0.
\]

Vectoriel :
\[
c_1+c_4>0,
\]

\[
G_V>0.
\]

Scalaire :
\[
K_S>0,
\qquad
G_S>0.
\]

Poisson :
\[
2-c_1-c_4\neq0.
\]

On vérifie qu'une région non vide existe sans fixer les couplages à des valeurs physiques.

Le témoin rationnel n'est qu'un certificat de non-vacuité.

In [11]:
# SC14.21 — Nonempty stable generic weak-field witness
witness = {
    c1: sp.Rational(1,10),
    c2: sp.Rational(1,5),
    c3: sp.Rational(3,10),
    c4: sp.Rational(2,5),
}

K_T = sp.factor(1-c1-c3)
K_V = sp.factor(c1+c4)
G_V = sp.factor((2*c1+c3**2-c1**2)/(2*(1-c1-c3)))

vals = {
    "K_T": sp.simplify(K_T.subs(witness)),
    "K_V": sp.simplify(K_V.subs(witness)),
    "G_V": sp.simplify(G_V.subs(witness)),
    "K_S": sp.simplify(K_S.subs(witness)),
    "G_S": sp.simplify(G_S.subs(witness)),
    "cT2": sp.simplify((1/K_T).subs(witness)),
    "cV2": sp.simplify((G_V/K_V).subs(witness)),
    "cS2": sp.simplify(cS2.subs(witness)),
    "c123": sp.simplify((c1+c2+c3).subs(witness)),
    "DeltaFF_numerator": sp.simplify((c1+3*c2+c3+2).subs(witness)),
    "two_minus_c14": sp.simplify((2-c1-c4).subs(witness)),
}

WEAK_FIELD_STABLE_GENERIC_SUBBRANCH_NONEMPTY = all([
    vals["K_T"] > 0,
    vals["K_V"] > 0,
    vals["G_V"] > 0,
    vals["K_S"] > 0,
    vals["G_S"] > 0,
    vals["c123"] != 0,
    vals["DeltaFF_numerator"] != 0,
    vals["two_minus_c14"] != 0,
])

assert WEAK_FIELD_STABLE_GENERIC_SUBBRANCH_NONEMPTY

print("witness =", witness)
for key,val in vals.items():
    print(key, "=", val)
print("WEAK_FIELD_STABLE_GENERIC_SUBBRANCH_NONEMPTY =", WEAK_FIELD_STABLE_GENERIC_SUBBRANCH_NONEMPTY)

witness = {c1: 1/10, c2: 1/5, c3: 3/10, c4: 2/5}
K_T = 3/5
K_V = 1/2
G_V = 7/30
K_S = 6
G_S = 6
cT2 = 5/3
cV2 = 7/15
cS2 = 1
c123 = 3/5
DeltaFF_numerator = 3
two_minus_c14 = 3/2
WEAK_FIELD_STABLE_GENERIC_SUBBRANCH_NONEMPTY = True


# SC14.22 — Verdict scientifique faible champ

Si toutes les étapes précédentes passent, le secteur linéarisé autour de Minkowski possède :

\[
\boxed{
5=2_T+2_V+1_S
}
\]

avec dispersions explicites dans les trois secteurs.

Le régime statique fournit :

\[
\boxed{
\nabla^2\Phi=4\pi G_{\rm eff}\rho
}
\]

et :

\[
\boxed{
G_{\rm eff}
=
\frac{G_0}{1-\frac12(c_1+c_4)}.
}
\]

Cela autorise un **PASS du benchmark faible champ sur une sous-branche ouverte stable**, mais pas sur tous les couplages.

Cela n'est pas encore :

- une validation PPN complète ;
- une confrontation observationnelle de \(c_T,c_V,c_S\) ;
- une preuve de Schwarzschild ;
- une autorisation de quantification.

In [12]:
# SC14.23 — Final weak-field classifier
SCALAR_LINEARIZED_BENCHMARK_PASS = all([
    SCALAR_QUADRATIC_ACTION_FULLY_DERIVED,
    SCALAR_LAPSE_SHIFT_CONSTRAINTS_SOLVED,
    SCALAR_REDUCED_PHYSICAL_ACTION_DERIVED,
    SCALAR_PROPAGATING_DOF_CLASSIFIED,
    SCALAR_DISPERSION_CLASSIFIED,
    SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED,
    FULL_LINEARIZED_DOF_MATCH_PASS,
])

WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH = all([
    UPSTREAM_GATE,
    SCALAR_LINEARIZED_BENCHMARK_PASS,
    STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
    POISSON_EQUATION_DERIVED,
    G_EFFECTIVE_DERIVED,
    WEAK_FIELD_STABLE_GENERIC_SUBBRANCH_NONEMPTY,
])

WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS = False

WEAK_FIELD_BENCHMARK_STATUS = (
    "PASS_ON_EXPLICIT_OPEN_STABLE_GENERIC_COUPLING_SUBBRANCH"
    if WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH
    else "PARTIAL_OR_FAIL_REPAIR_REQUIRED"
)

SCHWARZSCHILD_BENCHMARK_AUTHORIZED = WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH
CLASSICAL_PREDICTIONS_AUTHORIZED = False
PPN_BENCHMARK_COMPLETED = False
OBSERVATIONAL_MODE_SPEED_VIABILITY_CLASSIFIED = False
QUANTIZATION_READY = False

SC14_OBSTRUCTIONS = []
if not PPN_BENCHMARK_COMPLETED:
    SC14_OBSTRUCTIONS.append("PPN-NOT-YET-AUDITED")
if not OBSERVATIONAL_MODE_SPEED_VIABILITY_CLASSIFIED:
    SC14_OBSTRUCTIONS.append("OBSERVATIONAL-MODE-SPEED-VIABILITY-NOT-YET-AUDITED")

SC14_LOCAL_AUDIT_PASS = all([
    WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH,
    not WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS,
    SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
    not CLASSICAL_PREDICTIONS_AUTHORIZED,
    not PPN_BENCHMARK_COMPLETED,
    not QUANTIZATION_READY,
])

SC14_NEXT_AUTHORIZED = (
    "AUDIT-SCHWARZSCHILD-CLASSICAL-VACUUM-BENCHMARK"
    if SC14_LOCAL_AUDIT_PASS
    else "REPAIR-.3.3.14-SCALAR-POISSON-AUDIT"
)

assert SC14_LOCAL_AUDIT_PASS

print("SCALAR_LINEARIZED_BENCHMARK_PASS =", SCALAR_LINEARIZED_BENCHMARK_PASS)
print("FULL_LINEARIZED_DOF_MATCH_PASS =", FULL_LINEARIZED_DOF_MATCH_PASS)
print("SCALAR_SPEED_SQUARED =", SCALAR_SPEED_SQUARED)
print("G_EFFECTIVE =", G_eff)
print("WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH =", WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH)
print("WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS =", WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS)
print("WEAK_FIELD_BENCHMARK_STATUS =", WEAK_FIELD_BENCHMARK_STATUS)
print("SCHWARZSCHILD_BENCHMARK_AUTHORIZED =", SCHWARZSCHILD_BENCHMARK_AUTHORIZED)
print("CLASSICAL_PREDICTIONS_AUTHORIZED =", CLASSICAL_PREDICTIONS_AUTHORIZED)
print("SC14_OBSTRUCTIONS =", SC14_OBSTRUCTIONS)
print("SC14_NEXT_AUTHORIZED =", SC14_NEXT_AUTHORIZED)

SCALAR_LINEARIZED_BENCHMARK_PASS = True
FULL_LINEARIZED_DOF_MATCH_PASS = True
SCALAR_SPEED_SQUARED = (c1 + c2 + c3)*(c1 + c4 - 2)/((c1 + c4)*(c1 + c3 - 1)*(c1 + 3*c2 + c3 + 2))
G_EFFECTIVE = -2*G0/(c1 + c4 - 2)
WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH = True
WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS = False
WEAK_FIELD_BENCHMARK_STATUS = PASS_ON_EXPLICIT_OPEN_STABLE_GENERIC_COUPLING_SUBBRANCH
SCHWARZSCHILD_BENCHMARK_AUTHORIZED = True
CLASSICAL_PREDICTIONS_AUTHORIZED = False
SC14_OBSTRUCTIONS = ['PPN-NOT-YET-AUDITED', 'OBSERVATIONAL-MODE-SPEED-VIABILITY-NOT-YET-AUDITED']
SC14_NEXT_AUTHORIZED = AUDIT-SCHWARZSCHILD-CLASSICAL-VACUUM-BENCHMARK


# SC14.24 — Portée exacte du PASS

Le PASS local signifie :

\[
\boxed{
\text{Minkowski}
+
\text{faible champ linéarisé}
+
\text{Poisson}
}
\]

sont cohérents sur une sous-branche ouverte stable du secteur générique.

Le résultat central du secteur scalaire est :

\[
\boxed{
c_S^2=
\frac{
(c_1+c_2+c_3)(2-c_1-c_4)
}{
(c_1+c_4)(1-c_1-c_3)(c_1+3c_2+c_3+2)
}
}
\]

et :

\[
\boxed{
G_{\rm eff}
=
\frac{G_0}{1-\frac12(c_1+c_4)}.
}
\]

L'autorisation Schwarzschild signifie seulement que le prochain benchmark peut être construit ; elle ne préjuge pas de son résultat.

In [13]:
# SC14.25 — Machine-readable artifact
artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.14_Linearized_Scalar_Reduction_Poisson_and_G_Effective_Audit_FAST",
    "execution_scope": "LINEARIZED_SCALAR_REDUCTION_POISSON_AND_G_EFFECTIVE_AROUND_MINKOWSKI",
    "upstream": UPSTREAM,
    "scalar_sector": {
        "quadratic_action_fully_derived": SCALAR_QUADRATIC_ACTION_FULLY_DERIVED,
        "lapse_shift_constraints_solved": SCALAR_LAPSE_SHIFT_CONSTRAINTS_SOLVED,
        "reduced_action_crosscheck_pass": SCALAR_REDUCED_ACTION_CROSSCHECK_PASS,
        "kinetic_coefficient": str(SCALAR_KINETIC_COEFFICIENT),
        "gradient_coefficient": str(SCALAR_GRADIENT_COEFFICIENT),
        "speed_squared": str(SCALAR_SPEED_SQUARED),
        "propagating_dof": SCALAR_PROPAGATING_DOF,
        "second_class_reduction_materialized": SCALAR_SECOND_CLASS_REDUCTION_MATERIALIZED,
    },
    "dof": {
        "tensor": 2,
        "vector": 2,
        "scalar": SCALAR_PROPAGATING_DOF,
        "total": TOTAL_LINEARIZED_DOF,
        "full_match_pass": FULL_LINEARIZED_DOF_MATCH_PASS,
    },
    "static_poisson": {
        "bare_gravity_normalization": "S_grav=(16*pi*G0)^(-1) int N sqrt(h) [R3+KijKij-K^2+L_u]",
        "physical_potential": "Phi=n",
        "matter_linear_coupling": "-rho*n",
        "poisson_equation_derived": POISSON_EQUATION_DERIVED,
        "G_effective": str(G_eff),
        "G_effective_derived": G_EFFECTIVE_DERIVED,
        "static_weak_field_limit_classified": STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
    },
    "scientific_status": {
        "SCALAR_LINEARIZED_BENCHMARK_PASS": SCALAR_LINEARIZED_BENCHMARK_PASS,
        "WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH": WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH,
        "WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS": WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS,
        "WEAK_FIELD_BENCHMARK_STATUS": WEAK_FIELD_BENCHMARK_STATUS,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
        "CLASSICAL_PREDICTIONS_AUTHORIZED": CLASSICAL_PREDICTIONS_AUTHORIZED,
        "PPN_BENCHMARK_COMPLETED": PPN_BENCHMARK_COMPLETED,
        "OBSERVATIONAL_MODE_SPEED_VIABILITY_CLASSIFIED": OBSERVATIONAL_MODE_SPEED_VIABILITY_CLASSIFIED,
        "QUANTIZATION_READY": QUANTIZATION_READY,
    },
    "verdict": {
        "SC14_LOCAL_AUDIT_PASS": SC14_LOCAL_AUDIT_PASS,
        "obstructions_beyond_current_scope": SC14_OBSTRUCTIONS,
    },
    "next_authorized": SC14_NEXT_AUTHORIZED,
    "scope_note": "Weak-field benchmark passes on an explicit open stable generic coupling subbranch; PPN/observational viability and Schwarzschild remain separate downstream audits."
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.3.3.14_Linearized_Scalar_Reduction_Poisson_and_G_Effective_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
print("SC14 artifact =", artifact_path)

SC14 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.14_Linearized_Scalar_Reduction_Poisson_and_G_Effective_Audit_FAST.json


# Conclusion

`.3.3.14 FAST` ferme le dernier secteur faible champ si tous les tests passent.

Le prochain maillon autorisé devient alors :

\[
\boxed{
\texttt{AUDIT-SCHWARZSCHILD-CLASSICAL-VACUUM-BENCHMARK}.
}
\]

Schwarzschild reste un benchmark séparé : son autorisation n'est pas son PASS.